# 📒 Notebook 1 — ETL & Data Cleaning Pipeline
> **Peran:** Senior Data Scientist | **Proyek:** SPK Pantau Pasar Sumbawa

---

## 🗂️ 1. Struktur Proyek

```
SPK SIM/
├── data_set/               ← READ-ONLY: Raw Excel files (jangan diubah!)
│   ├── PIHPS/
│   ├── SP2KP/
│   ├── Diskoperindag Sumbawa/
│   └── UPT Pasar Seketeng/
├── notebook/               ← Jupyter Notebooks (pipeline ETL)
└── processed_data/         ← Output (auto-created)
    ├── cleaned/            ← Hasil cleaning per sumber
    ├── merged/             ← Dataset gabungan
    ├── processed/          ← Dataset dengan kalender lengkap
    └── features/           ← Dataset final siap modeling
```

---

## 🧹 2. Mengapa Data Cleaning Penting?

Data harga bahan pokok dikumpulkan dari **4 sumber berbeda** yang masing-masing memiliki:

| Sumber | Format | Frekuensi | Tantangan |
|--------|--------|-----------|-----------|
| **PIHPS** | Wide → kolom = tanggal | Harian (kerja) | Harga string berkoma: `"13,000"`, tanda `-` untuk kosong |
| **SP2KP** | Wide → kolom = tanggal | Harian (kerja) | 2 baris metadata di atas header |
| **Diskoperindag** | 2 layout berbeda (2023 vs 2026) | Bulanan | Nomor hari = kolom, bulan/tahun dari nama file |
| **UPT Seketeng** | Long (mingguan) | Mingguan | Tanggal dalam tanda kurung siku: `[28/12/2025]` |

Proses cleaning menghasilkan format seragam:

```
Tanggal  |  Komoditas        |  Harga   |  Sumber
---------|-------------------|----------|----------
2024-01-02 | beras_medium    | 14000.0  | SP2KP
2024-01-02 | cabai_merah_besar| 40000.0 | PIHPS
```

---

## 🔑 3. Standardisasi Komoditas

Nama komoditas mentah sangat beragam antar sumber. Fungsi `standardize_commodity()` menangani hal ini dengan:
1. **Normalisasi teks** → huruf kecil, tanpa spasi berlebih.
2. **Ganti delimiter** (koma, garis miring, tanda hubung) → spasi.
3. **Hapus simbol** non-alfanumerik.
4. **Dictionary mapping** → memetakan alias ke label standar.

**Contoh:**
- `"BERAS MEDIUM"` → `"beras_medium"`
- `"Beras Cap Dua Jempol"` → `"beras_premium"`
- `"Cabai Merah Besar,1 kg"` → `"cabai_merah_besar"`
- `"Kedelai Lokal/Kuning Kecil"` → `"kedelai_lokal"`


In [1]:
import os
import re
import logging
from pathlib import Path
from typing import List, Dict

import pandas as pd
import numpy as np

# ── Logging Setup ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Cleaning")

# ── Path Definitions (Pathlib) ────────────────────────────────────────────────
BASE_PATH: Path = Path("../data_set")
PROCESSED_DIR: Path = Path("../processed_data")
CLEANED_DIR: Path = PROCESSED_DIR / "cleaned"

# ── Auto-create Output Directories ───────────────────────────────────────────
for _dir in [
    CLEANED_DIR,
    PROCESSED_DIR / "merged",
    PROCESSED_DIR / "processed",
    PROCESSED_DIR / "features",
]:
    _dir.mkdir(parents=True, exist_ok=True)

logger.info("✅ Directory tree ready under: %s", PROCESSED_DIR.resolve())


10:43:36 [INFO] ✅ Directory tree ready under: C:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\processed_data


In [2]:
# ── Load Commodity Mapping Dictionary from JSON ───────────────────────────────
import json
import re

_mapping_file = Path("commodity_mapping.json")
with open(_mapping_file, "r", encoding="utf-8") as _f:
    COMMODITY_MAPPING: dict[str, str] = json.load(_f)

logger.info("Kamus komoditas dimuat: %d entri dari %s", len(COMMODITY_MAPPING), _mapping_file.name)


def standardize_commodity(name: str) -> str:
    """Standardize a raw commodity name to a canonical snake_case key.

    Steps:
        1. Replace common delimiters (comma, slash, hyphen) with a space.
        2. Lowercase and strip leading/trailing whitespace.
        3. Remove all characters that are not alphanumeric or space.
        4. Collapse multiple spaces/underscores into a single underscore.
        5. Look up the result in COMMODITY_MAPPING; return as-is if not found.

    Args:
        name: Raw commodity name string from the source dataset.

    Returns:
        Standardized snake_case commodity key string.

    Examples:
        >>> standardize_commodity("Beras Cap Dua Jempol")
        'beras_premium'
        >>> standardize_commodity("Cabai Merah Besar,1 kg")
        'cabai_merah_besar'
    """
    if not isinstance(name, str):
        return ""
    cleaned: str = (
        name.replace(",", " ").replace("/", " ").replace("-", " ").lower().strip()
    )
    cleaned = re.sub(r"[^a-z0-9\s]", "", cleaned)
    cleaned = re.sub(r"[\s_]+", "_", cleaned)
    cleaned = cleaned.strip("_")
    return COMMODITY_MAPPING.get(cleaned, cleaned)


# Quick smoke test (ASCII-safe output)
_examples = [
    ("BERAS MEDIUM", "beras_medium"),
    ("Beras Cap Dua Jempol", "beras_premium"),
    ("Cabai Merah Besar,1 kg", "cabai_merah_besar"),
    ("Kedelai Lokal/Kuning Kecil", "kedelai_lokal"),
    ("Minyak KITA Kemasan Botol", "minyakita"),
]
print("Standardisasi komoditas -- smoke test:")
print("-" * 60)
for raw, expected in _examples:
    result = standardize_commodity(raw)
    status = "[OK]" if result == expected else "[FAIL]"
    print(f"  {status}  '{raw}'  ->  '{result}'")


10:43:36 [INFO] Kamus komoditas dimuat: 154 entri dari commodity_mapping.json


Standardisasi komoditas -- smoke test:
------------------------------------------------------------
  [OK]  'BERAS MEDIUM'  ->  'beras_medium'
  [OK]  'Beras Cap Dua Jempol'  ->  'beras_premium'
  [OK]  'Cabai Merah Besar,1 kg'  ->  'cabai_merah_besar'
  [OK]  'Kedelai Lokal/Kuning Kecil'  ->  'kedelai_lokal'
  [OK]  'Minyak KITA Kemasan Botol'  ->  'minyakita'


---
## 📦 4. Modul Cleaning: PIHPS

**Sumber:** `data_set/PIHPS/`  
**Format:** Wide table — kolom berisi tanggal bertipe string `"dd/ mm/ yyyy"`.  
**Output:** `processed_data/cleaned/pihps_clean.csv`

### Alur ETL PIHPS
```
ExcelFile (wide)
    → Melt (wide→long)
    → Filter baris kategori Roman (I, II, III, ...)
    → Parse tanggal ("dd/ mm/ yyyy" → datetime)
    → Bersihkan harga ("13,000" → 13000.0, "-" → NaN)
    → Standardisasi nama komoditas
    → Label kolom Sumber = "PIHPS"
    → Export CSV
```


In [3]:
def clean_pihps(folder_path: Path, output_file: Path) -> pd.DataFrame:
    """Clean all PIHPS Excel files and export a unified long-format DataFrame.

    PIHPS files have a wide layout where each date occupies its own column,
    formatted as "dd/ mm/ yyyy". Prices are stored as comma-separated strings
    (e.g. "13,000"). Category header rows marked with Roman numerals are dropped.

    Args:
        folder_path: Directory containing PIHPS .xlsx files.
        output_file: Destination path for the cleaned CSV output.

    Returns:
        Cleaned DataFrame with columns [Tanggal, Komoditas, Harga, Sumber].
    """
    all_dfs: List[pd.DataFrame] = []
    roman_re = re.compile(r"^[ivxlcdm]+$", re.IGNORECASE)

    for f in sorted(folder_path.glob("*.xlsx")):
        logger.info("PIHPS ← %s", f.name)
        xl = pd.ExcelFile(f)
        df = xl.parse(xl.sheet_names[0])

        cols = list(df.columns)
        no_col, kom_col, date_cols = cols[0], cols[1], cols[2:]

        # Wide → Long
        df_melted = df.melt(
            id_vars=[no_col, kom_col],
            value_vars=date_cols,
            var_name="Tanggal_Raw",
            value_name="Harga_Raw",
        )

        # Drop Roman-numeral category header rows
        df_melted = df_melted[
            ~df_melted[no_col].astype(str).str.strip().str.match(roman_re, na=False)
        ]

        # Parse tanggal: strip spaces → "dd/mm/yyyy"
        df_melted["Tanggal"] = pd.to_datetime(
            df_melted["Tanggal_Raw"].astype(str).str.replace(" ", "", regex=False),
            format="%d/%m/%Y",
            errors="coerce",
        )

        # Parse harga: remove commas, convert "-" to NaN
        harga_str = (
            df_melted["Harga_Raw"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
            .replace("-", pd.NA)
        )
        df_melted["Harga"] = pd.to_numeric(harga_str, errors="coerce")

        # Drop invalid rows
        df_melted.dropna(subset=["Tanggal", "Harga"], inplace=True)

        # Standardize commodity & label source
        df_melted["Komoditas"] = df_melted[kom_col].apply(standardize_commodity)
        df_melted["Sumber"] = "PIHPS"

        all_dfs.append(df_melted[["Tanggal", "Komoditas", "Harga", "Sumber"]])

    result = pd.concat(all_dfs, ignore_index=True)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    # Deduplikasi per sumber sebelum ekspor (audit metodologi: cleaning hanya per-sumber)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    result.to_csv(output_file, index=False)
    logger.info("✅ PIHPS selesai. Shape: %s → %s", result.shape, output_file.name)
    return result


df_pihps = clean_pihps(BASE_PATH / "PIHPS", CLEANED_DIR / "pihps_clean.csv")
df_pihps.head()


10:43:36 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2021_PIHPS.xlsx
10:43:37 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2022_PIHPS.xlsx
10:43:37 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2023_PIHPS.xlsx
10:43:37 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2024_PIHPS.xlsx
10:43:38 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2025_PIHPS.xlsx
10:43:38 [INFO] PIHPS ← Tabel Harga Berdasarkan Daerah 2026_PIHPS.xlsx
10:43:38 [INFO] ✅ PIHPS selesai. Shape: (20940, 4) → pihps_clean.csv


,Tanggal,Komoditas,Harga,Sumber
0,2021-01-04,beras_medium,10000.0,PIHPS
2,2021-01-04,beras_premium,12000.0,PIHPS
4,2021-01-04,daging_ayam_ras,41000.0,PIHPS
5,2021-01-04,daging_sapi_paha_belakang,120000.0,PIHPS
6,2021-01-04,daging_sapi_paha_depan,120000.0,PIHPS


---
## 📦 5. Modul Cleaning: SP2KP

**Sumber:** `data_set/SP2KP/`  
**Dataset Anchor Utama** — digunakan sebagai referensi prioritas tertinggi.  
**Format:** Wide table dalam sheet `Average Prices`. Terdapat **2 baris metadata** di atas header.  
**Output:** `processed_data/cleaned/sp2kp_clean.csv`

### Alur ETL SP2KP
```
ExcelFile (sheet "Average Prices", skiprows=2)
    → Set baris ke-1 sebagai header kolom
    → Melt (wide→long) pada kolom tanggal
    → Parse tanggal (format ISO: "YYYY-MM-DD")
    → Cast harga ke float, drop 0 dan NaN
    → Standardisasi komoditas
    → Label Sumber = "SP2KP"
    → Export CSV
```


In [4]:
def clean_sp2kp(folder_path: Path, output_file: Path) -> pd.DataFrame:
    """Clean all SP2KP Excel files and export a unified long-format DataFrame.

    SP2KP files use an 'Average Prices' sheet with 2 rows of metadata above the
    actual column headers. Date columns are already in ISO-8601 format (YYYY-MM-DD).

    Args:
        folder_path: Directory containing SP2KP .xlsx files.
        output_file: Destination path for the cleaned CSV output.

    Returns:
        Cleaned DataFrame with columns [Tanggal, Komoditas, Harga, Sumber].
    """
    all_dfs: List[pd.DataFrame] = []

    for f in sorted(folder_path.glob("*.xlsx")):
        logger.info("SP2KP ← %s", f.name)
        xl = pd.ExcelFile(f)
        sheet_name = (
            "Average Prices" if "Average Prices" in xl.sheet_names else xl.sheet_names[0]
        )
        df = xl.parse(sheet_name, skiprows=2)

        # Row 0 contains the actual column headers
        df.columns = df.iloc[0].astype(str).str.strip()
        df = df.iloc[1:].copy()

        cols = list(df.columns)
        var_col, qty_col, unit_col, date_cols = cols[0], cols[1], cols[2], cols[3:]

        # Wide → Long
        df_melted = df.melt(
            id_vars=[var_col, qty_col, unit_col],
            value_vars=date_cols,
            var_name="Tanggal_Raw",
            value_name="Harga_Raw",
        )

        # Parse fields
        df_melted["Tanggal"] = pd.to_datetime(
            df_melted["Tanggal_Raw"].astype(str).str.strip(), errors="coerce"
        )
        df_melted["Harga"] = pd.to_numeric(df_melted["Harga_Raw"], errors="coerce")

        # Drop invalid and zero-price rows
        df_melted.dropna(subset=["Tanggal", "Harga"], inplace=True)
        df_melted = df_melted[df_melted["Harga"] > 0]

        # Standardize commodity & label source
        df_melted["Komoditas"] = df_melted[var_col].apply(standardize_commodity)
        df_melted["Sumber"] = "SP2KP"

        all_dfs.append(df_melted[["Tanggal", "Komoditas", "Harga", "Sumber"]])

    result = pd.concat(all_dfs, ignore_index=True)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    # Deduplikasi per sumber sebelum ekspor (audit metodologi: cleaning hanya per-sumber)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    result.to_csv(output_file, index=False)
    logger.info("✅ SP2KP selesai. Shape: %s → %s", result.shape, output_file.name)
    return result


df_sp2kp = clean_sp2kp(BASE_PATH / "SP2KP", CLEANED_DIR / "sp2kp_clean.csv")
df_sp2kp.head()


10:43:38 [INFO] SP2KP ← Statistik Harian Agustus 2024 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian Agustus 2025 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian April 2024 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian April 2025 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian April 2026 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian Desember 2024 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian Desember 2025 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian Februari 2024 Pasar Seketeng.xlsx
10:43:38 [INFO] SP2KP ← Statistik Harian Februari 2025 Pasar Seketeng.xlsx
10:43:39 [INFO] SP2KP ← Statistik Harian Februari 2026 Pasar Seketeng.xlsx
10:43:39 [INFO] SP2KP ← Statistik Harian Januari 2025 Pasar Seketeng.xlsx
10:43:39 [INFO] SP2KP ← Statistik Harian Januari 2026 Pasar Seketeng.xlsx
10:43:39 [INFO] SP2KP ← Statistik Harian Juli 2024 Pasar Seketeng.xlsx
10:43:39 [INFO] SP2KP ← Statistik Harian J

,Tanggal,Komoditas,Harga,Sumber
0,2024-08-01,beras_medium,14000.0,SP2KP
1,2024-08-01,beras_premium,15000.0,SP2KP
2,2024-08-01,beras_sphp_bulog,12000.0,SP2KP
3,2024-08-01,kedelai_impor,18000.0,SP2KP
4,2024-08-01,cabai_merah_keriting,45000.0,SP2KP


---
## 📦 6. Modul Cleaning: Diskoperindag Sumbawa

**Sumber:** `data_set/Diskoperindag Sumbawa/`  
**Format:** Terdapat **2 layout berbeda** tergantung tahun file.  
**Output:** `processed_data/cleaned/diskoperindag_clean.csv`

### Dua Layout yang Terdeteksi Otomatis

| Layout | Tahun | Ciri Khas | Deteksi |
|--------|-------|-----------|---------|
| **2026** | Jan-Jun 2026 | Header 1 baris: `No. | SubVariant | Kuantitas | Satuan | 1 | 2 | 3 ...` | Kolom `SubVariant` ada |
| **2023/2024** | Jan-Mar 2023, Jan-Feb 2024 | Header 2 baris: baris pertama berisi `No. | Nama ... | Satuan | Tanggal`, baris kedua berisi nomor hari | Kolom `Tanggal` ada |

Bulan dan Tahun **diekstrak dari nama file** (contoh: `"... Apr 2026_Diskoperindag.xlsx"` → Month=4, Year=2026), kemudian digabung dengan nomor hari untuk membentuk tanggal penuh.


In [5]:
def clean_diskoperindag(folder_path: Path, output_file: Path) -> pd.DataFrame:
    """Clean all Diskoperindag Excel files and export a unified long-format DataFrame.

    Handles two distinct file layouts automatically:
      - Layout '2026': single header row with 'SubVariant' label.
      - Layout '2023': double header row where the second row contains day numbers
        and the first row contains a 'Tanggal' cell.

    Month and Year are extracted from the filename to construct full dates.

    Args:
        folder_path: Directory containing Diskoperindag .xlsx files.
        output_file: Destination path for the cleaned CSV output.

    Returns:
        Cleaned DataFrame with columns [Tanggal, Komoditas, Harga, Sumber].
    """
    all_dfs: List[pd.DataFrame] = []
    month_map: Dict[str, int] = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "jun": 6,
        "jul": 7, "agu": 8, "sep": 9, "okt": 10, "nov": 11, "des": 12,
        "agt": 8, "nop": 11,
    }

    for f in sorted(folder_path.glob("*.xlsx")):
        # ── Extract month/year from filename ──────────────────────────────
        stem = f.name.replace("_Diskoperindag.xlsx", "").replace(".xlsx", "")
        parts = stem.split(" ")
        month: int = month_map.get(parts[-2].lower()[:3], 1)
        year: int = int(parts[-1])
        logger.info("Diskoperindag ← %s  (Bulan=%d, Tahun=%d)", f.name, month, year)

        df_raw = pd.ExcelFile(f).parse(
            pd.ExcelFile(f).sheet_names[0], header=None
        )

        # ── Auto-detect layout ────────────────────────────────────────────
        header_row_idx: int = -1
        layout_type: str = "2026"
        for idx, row in df_raw.iterrows():
            vals_lower = [str(v).strip().lower() for v in row.values if pd.notna(v)]
            if any(n in vals_lower for n in ["no.", "no"]):
                if "tanggal" in vals_lower:
                    header_row_idx, layout_type = idx, "2023"
                elif "subvariant" in vals_lower or any(
                    v.replace(".0", "").isdigit() for v in vals_lower
                ):
                    header_row_idx, layout_type = idx, "2026"
                break
        if header_row_idx == -1:
            header_row_idx, layout_type = 2, "2026"
        logger.info("  Layout '%s' terdeteksi di baris %d", layout_type, header_row_idx)

        # ── Parse based on layout ─────────────────────────────────────────
        if layout_type == "2026":
            df = df_raw.iloc[header_row_idx:].copy()
            df.columns = df.iloc[0].astype(str).str.strip()
            df = df.iloc[1:]
            df.columns = [
                c[:-2] if str(c).endswith(".0") and str(c)[:-2].isdigit() else str(c)
                for c in df.columns
            ]
            no_col, var_col, qty_col, unit_col = (
                df.columns[0], df.columns[1], df.columns[2], df.columns[3]
            )
            day_cols = [c for c in df.columns[4:] if c.replace(".0", "").isdigit()]
            df_melted = df.melt(
                id_vars=[no_col, var_col, qty_col, unit_col],
                value_vars=day_cols,
                var_name="Hari_Raw",
                value_name="Harga_Raw",
            )

        else:  # layout 2023
            df = df_raw.iloc[header_row_idx:].copy()
            raw_days = list(df.iloc[1].values)
            new_cols: List[str] = []
            for i, (c, d) in enumerate(zip(df.iloc[0].values, raw_days)):
                if i < 3:
                    new_cols.append(str(c).strip())
                else:
                    ds = str(d).strip()
                    new_cols.append(ds[:-2] if ds.endswith(".0") else ds)
            df.columns = new_cols
            df = df.iloc[2:]
            no_col, var_col, unit_col = df.columns[0], df.columns[1], df.columns[2]
            day_cols = [c for c in df.columns[3:] if c.replace(".0", "").isdigit()]
            df_melted = df.melt(
                id_vars=[no_col, var_col, unit_col],
                value_vars=day_cols,
                var_name="Hari_Raw",
                value_name="Harga_Raw",
            )

        # ── Build full date from day number + month + year ────────────────
        df_melted["Hari"] = pd.to_numeric(df_melted["Hari_Raw"], errors="coerce")
        df_melted.dropna(subset=["Hari"], inplace=True)
        df_melted["Hari"] = df_melted["Hari"].astype(int)

        def _make_date(row: pd.Series) -> pd.Timestamp:
            try:
                return pd.Timestamp(year=year, month=month, day=int(row["Hari"]))
            except Exception:
                return pd.NaT

        df_melted["Tanggal"] = df_melted.apply(_make_date, axis=1)
        df_melted.dropna(subset=["Tanggal"], inplace=True)

        # ── Parse price ───────────────────────────────────────────────────
        df_melted["Harga"] = pd.to_numeric(df_melted["Harga_Raw"], errors="coerce")
        df_melted.dropna(subset=["Harga"], inplace=True)
        df_melted = df_melted[df_melted["Harga"] > 0]

        # Standardize commodity & label source
        df_melted["Komoditas"] = df_melted[var_col].apply(standardize_commodity)
        df_melted["Sumber"] = "Diskoperindag"
        all_dfs.append(df_melted[["Tanggal", "Komoditas", "Harga", "Sumber"]])

    result = pd.concat(all_dfs, ignore_index=True)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    # Deduplikasi per sumber sebelum ekspor (audit metodologi: cleaning hanya per-sumber)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    result.to_csv(output_file, index=False)
    logger.info(
        "✅ Diskoperindag selesai. Shape: %s → %s", result.shape, output_file.name
    )
    return result


df_disko = clean_diskoperindag(
    BASE_PATH / "Diskoperindag Sumbawa", CLEANED_DIR / "diskoperindag_clean.csv"
)
df_disko.head()


10:43:40 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Apr 2026_Diskoperindag.xlsx  (Bulan=4, Tahun=2026)
10:43:40 [INFO]   Layout '2026' terdeteksi di baris 3
10:43:40 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Feb 2023_Diskoperindag.xlsx  (Bulan=2, Tahun=2023)
10:43:40 [INFO]   Layout '2023' terdeteksi di baris 9
10:43:40 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Feb 2024_Diskoperindag.xlsx  (Bulan=2, Tahun=2024)
10:43:40 [INFO]   Layout '2023' terdeteksi di baris 9
10:43:41 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Feb 2026_Diskoperindag.xlsx  (Bulan=2, Tahun=2026)
10:43:41 [INFO]   Layout '2026' terdeteksi di baris 3
10:43:41 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Jan 2023_Diskoperindag.xlsx  (Bulan=1, Tahun=2023)
10:43:41 [INFO]   Layout '2023' terdeteksi di baris 9
10:43:41 [INFO] Diskoperindag ← Tabel Harga Berdasarkan Daerah Jan 2024_Diskoperindag.xlsx  (Bulan=1, Tahun=2024)
10:43:41 [INFO]   Layout '2023' terdeteksi di 

,Tanggal,Komoditas,Harga,Sumber
0,2026-04-01,beras_premium,13400.0,Diskoperindag
1,2026-04-01,beras_medium,13400.0,Diskoperindag
4,2026-04-01,beras_sphp_bulog,12200.0,Diskoperindag
5,2026-04-01,kedelai_impor,15000.0,Diskoperindag
6,2026-04-01,cabai_merah_keriting,45000.0,Diskoperindag


---
## 📦 7. Modul Cleaning: UPT Pasar Seketeng

**Sumber:** `data_set/UPT Pasar Seketeng/`  
**Format:** Long table mingguan. Tanggal berada dalam tanda kurung siku: `[28/12/2025]`.  
**Output:** `processed_data/cleaned/upt_clean.csv`

### Alur ETL UPT
```
ExcelFile (sheet "Dataset YYYY", skiprows=2)
    → Hapus kurung siku dari string tanggal
    → Parse tanggal (dayfirst=True)
    → Cast harga ke float, drop 0 dan NaN
    → Standardisasi komoditas
    → Label Sumber = "UPT_Seketeng"
    → Export CSV
```


In [6]:
def clean_upt(folder_path: Path, output_file: Path) -> pd.DataFrame:
    """Clean all UPT Pasar Seketeng Excel files and export a unified long-format DataFrame.

    UPT files contain weekly price data in long format. Date strings include
    square brackets (e.g. '[28/12/2025]') which are stripped before parsing.

    Unit Normalization (cols[5]):
        UPT records egg prices per butir (piece), while all other sources use per kg.
        Rows where satuan == 'butir' are converted using standard market equivalences:
          - telur_ayam_ras      : x 16 butir/kg
          - telur_ayam_kampung  : x 18 butir/kg
          - telur_itik          : x 15 butir/kg

    Args:
        folder_path: Directory containing UPT .xlsx files.
        output_file: Destination path for the cleaned CSV output.

    Returns:
        Cleaned DataFrame with columns [Tanggal, Komoditas, Harga, Sumber].
    """
    # Conversion factors: butir (pieces) per kg for each egg commodity
    BUTIR_PER_KG: dict[str, float] = {
        "telur_ayam_ras":      16.0,  # ~60-62 g/butir → 1 kg ≈ 16 butir
        "telur_ayam_kampung": 18.0,  # ~55-57 g/butir → 1 kg ≈ 18 butir
        "telur_itik":          15.0,  # ~65-70 g/butir → 1 kg ≈ 15 butir
    }

    all_dfs: List[pd.DataFrame] = []

    for f in sorted(folder_path.glob("*.xlsx")):
        logger.info("UPT ← %s", f.name)
        xl = pd.ExcelFile(f)
        sheet_name = next(s for s in xl.sheet_names if "Dataset" in s)
        df = xl.parse(sheet_name, skiprows=2)

        cols = list(df.columns)
        date_col, kom_col, unit_col, price_col = cols[1], cols[4], cols[5], cols[6]

        # Strip brackets from date string: [DD/MM/YYYY] → DD/MM/YYYY
        df["Tanggal_Str"] = (
            df[date_col]
            .astype(str)
            .str.replace("[", "", regex=False)
            .str.replace("]", "", regex=False)
            .str.strip()
        )
        df["Tanggal"] = pd.to_datetime(df["Tanggal_Str"], dayfirst=True, errors="coerce")
        df["Harga"] = pd.to_numeric(df[price_col], errors="coerce")

        df.dropna(subset=["Tanggal", "Harga"], inplace=True)
        df = df[df["Harga"] > 0]

        df["Komoditas"] = df[kom_col].apply(standardize_commodity)
        df["Sumber"] = "UPT_Seketeng"

        # ── Unit normalization: butir → per kg ────────────────────────────────
        satuan_raw = df[unit_col].astype(str).str.strip().str.lower()
        butir_mask = satuan_raw == "butir"
        for komoditas_key, factor in BUTIR_PER_KG.items():
            apply_mask = butir_mask & (df["Komoditas"] == komoditas_key)
            if apply_mask.any():
                df.loc[apply_mask, "Harga"] = df.loc[apply_mask, "Harga"] * factor
                logger.info(
                    "  Unit fix: %s (%d baris) butir → /kg (x%.0f) @ %s",
                    komoditas_key, apply_mask.sum(), factor, f.name,
                )

        all_dfs.append(df[["Tanggal", "Komoditas", "Harga", "Sumber"]])

    result = pd.concat(all_dfs, ignore_index=True)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    # Deduplikasi per sumber sebelum ekspor (audit metodologi: cleaning hanya per-sumber)
    result.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first", inplace=True)
    result.to_csv(output_file, index=False)
    logger.info("✅ UPT selesai. Shape: %s → %s", result.shape, output_file.name)
    return result


df_upt = clean_upt(BASE_PATH / "UPT Pasar Seketeng", CLEANED_DIR / "upt_clean.csv")
df_upt.head()


10:43:41 [INFO] UPT ← Dataset_Komoditas_2025.xlsx
C:\Users\Saputra Budiman\AppData\Local\Temp\ipykernel_19316\4264049360.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Tanggal"] = pd.to_datetime(df["Tanggal_Str"], dayfirst=True, errors="coerce")
10:43:42 [INFO]   Unit fix: telur_ayam_ras (8 baris) butir → /kg (x16) @ Dataset_Komoditas_2025.xlsx
10:43:42 [INFO]   Unit fix: telur_ayam_kampung (8 baris) butir → /kg (x18) @ Dataset_Komoditas_2025.xlsx
10:43:42 [INFO]   Unit fix: telur_itik (8 baris) butir → /kg (x15) @ Dataset_Komoditas_2025.xlsx
10:43:42 [INFO] UPT ← Dataset_Komoditas_2026.xlsx
C:\Users\Saputra Budiman\AppData\Local\Temp\ipykernel_19316\4264049360.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify

,Tanggal,Komoditas,Harga,Sumber
0,2025-11-03,beras_medium,15500.0,UPT_Seketeng
2,2025-11-03,jagung_pipilan_kuning,8000.0,UPT_Seketeng
3,2025-11-03,kedelai_lokal,20000.0,UPT_Seketeng
4,2025-11-03,ketela_pohon,6000.0,UPT_Seketeng
5,2025-11-03,ubi_jalar,10000.0,UPT_Seketeng


---
## ✅ 8. Ringkasan Hasil Cleaning


In [7]:
# ── Summary of all four cleaned datasets ─────────────────────────────────────
summary = {
    "PIHPS":         df_pihps,
    "SP2KP":         df_sp2kp,
    "Diskoperindag": df_disko,
    "UPT_Seketeng":  df_upt,
}

print(f"{'Sumber':<16} {'Baris':>8} {'Komoditas Unik':>16} {'Min Tanggal':<14} {'Max Tanggal'}")
print("-" * 70)
for name, df in summary.items():
    n_rows      = len(df)
    n_kom       = df["Komoditas"].nunique()
    min_date    = df["Tanggal"].min().date()
    max_date    = df["Tanggal"].max().date()
    print(f"{name:<16} {n_rows:>8,} {n_kom:>16} {str(min_date):<14} {max_date}")

print()
print("📁 File output tersimpan di:", CLEANED_DIR.resolve())


Sumber              Baris   Komoditas Unik Min Tanggal    Max Tanggal
----------------------------------------------------------------------
PIHPS              20,940               15 2021-01-04     2026-06-19
SP2KP              24,445               44 2024-02-01     2026-06-18
Diskoperindag       7,963               46 2023-01-02     2026-06-15
UPT_Seketeng          601               26 2025-11-03     2026-11-10

📁 File output tersimpan di: C:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\processed_data\cleaned
